# Transformer

In [47]:
import warnings, math
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import torch
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from lightning.pytorch import Trainer, seed_everything

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss, MAE, RMSE
from pytorch_forecasting.data import NaNLabelEncoder
from pytorch_forecasting.data.encoders import TorchNormalizer


## Load the CSV and preprocess the data

In [68]:
seed_everything(42)
PATH = "/home/wqiaojoe/Development/ml-model-battery-levels-iot/src/data_cleaning/4e0031000251353337353037_cleaned_data.csv"
df = pd.read_csv(PATH)
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

df["date"] = pd.to_datetime(df.date)
df = df.sort_values("date").reset_index(drop=True)
# Add a single-series id (if you only have one device)
df["series_id"] = "device_1"

# Build an integer time index starting at 0 (TFT requires an int 'time_idx')
df["time_idx"] = (df["date"] - df["date"].min()).dt.total_seconds() // 60
df["time_idx"] = df["time_idx"].astype(int)
df.head(10)

Seed set to 42


,date,barometric_pressure,battery,humidity,rain_meter,soil_moisture,temperature,wind_speed,day_length,series_id,time_idx
0,2018-05-10 14:28:00,1006.35,91.65,36.34,0.00,0.05,29.31,0.0,857,device_1,0
1,2018-05-10 14:38:00,1006.20,91.65,35.24,0.00,57.61,28.29,0.0,857,device_1,10
2,2018-05-10 14:40:00,1006.31,91.65,51.07,0.00,49.04,28.04,0.0,857,device_1,12
3,2018-05-10 14:41:00,1006.36,91.65,36.98,0.00,0.10,28.20,0.0,857,device_1,13
4,2018-05-10 14:46:00,1006.33,90.54,35.70,0.00,74.54,28.99,0.0,857,device_1,18
5,2018-05-10 14:57:00,1006.20,88.96,34.12,0.00,0.00,27.71,0.0,857,device_1,29
6,2018-05-10 15:05:00,1006.26,87.93,36.78,0.28,76.20,26.78,0.0,857,device_1,37
7,2018-05-10 15:06:00,1006.18,87.93,37.40,0.00,63.01,26.71,0.0,857,device_1,38
8,2018-05-10 15:11:00,1006.19,87.49,36.70,0.00,80.27,27.22,0.0,857,device_1,43
9,2018-05-10 15:12:00,1006.15,87.42,36.19,0.00,78.94,27.28,0.0,857,device_1,44


## Feature engineering

In [ ]:
# Calendar/time features known into the future (help model seasonality)
df["hour"] = df["date"].dt.hour
df["dow"]  = df["date"].dt.dayofweek
df["dom"]  = df["date"].dt.day

# Optional cyclic encodings
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["dow_sin"]  = np.sin(2 * np.pi * df["dow"] / 7)
df["dow_cos"]  = np.cos(2 * np.pi * df["dow"] / 7)

# Reorder columns (optional)
df = df[[
    "series_id","time_idx","date",
    "battery",
    "barometric_pressure","humidity","rain_meter","soil_moisture","temperature","wind_speed","day_length",
    "hour","dow","dom","hour_sin","hour_cos","dow_sin","dow_cos"
]]
df.head(10)

,series_id,time_idx,date,battery,barometric_pressure,humidity,rain_meter,soil_moisture,temperature,wind_speed,day_length,hour,dow,dom,hour_sin,hour_cos,dow_sin,dow_cos
0,device_1,0,2018-05-10 14:28:00,91.65,1006.35,36.34,0.00,0.05,29.31,0.0,857,14,3,10,-0.500000,-0.866025,0.433884,-0.900969
1,device_1,10,2018-05-10 14:38:00,91.65,1006.20,35.24,0.00,57.61,28.29,0.0,857,14,3,10,-0.500000,-0.866025,0.433884,-0.900969
2,device_1,12,2018-05-10 14:40:00,91.65,1006.31,51.07,0.00,49.04,28.04,0.0,857,14,3,10,-0.500000,-0.866025,0.433884,-0.900969
3,device_1,13,2018-05-10 14:41:00,91.65,1006.36,36.98,0.00,0.10,28.20,0.0,857,14,3,10,-0.500000,-0.866025,0.433884,-0.900969
4,device_1,18,2018-05-10 14:46:00,90.54,1006.33,35.70,0.00,74.54,28.99,0.0,857,14,3,10,-0.500000,-0.866025,0.433884,-0.900969
5,device_1,29,2018-05-10 14:57:00,88.96,1006.20,34.12,0.00,0.00,27.71,0.0,857,14,3,10,-0.500000,-0.866025,0.433884,-0.900969
6,device_1,37,2018-05-10 15:05:00,87.93,1006.26,36.78,0.28,76.20,26.78,0.0,857,15,3,10,-0.707107,-0.707107,0.433884,-0.900969
7,device_1,38,2018-05-10 15:06:00,87.93,1006.18,37.40,0.00,63.01,26.71,0.0,857,15,3,10,-0.707107,-0.707107,0.433884,-0.900969
8,device_1,43,2018-05-10 15:11:00,87.49,1006.19,36.70,0.00,80.27,27.22,0.0,857,15,3,10,-0.707107,-0.707107,0.433884,-0.900969
9,device_1,44,2018-05-10 15:12:00,87.42,1006.15,36.19,0.00,78.94,27.28,0.0,857,15,3,10,-0.707107,-0.707107,0.433884,-0.900969


In [75]:
# Forecast horizon (how far ahead you predict) and encoder length (history window)
max_prediction_length = 1   # 1 steps ahead — change as needed
max_encoder_length    = 120  # 120 steps of history (20 hours at 10-min interval) — change as needed

# Simple time-based split: last chunk for validation
validation_start = df["time_idx"].max() - 200  # ~100 windows for val


In [76]:
print(validation_start)
print(df.time_idx.max())
print(validation_start-max_encoder_length)

179883
180083
179763


In [77]:
df.columns

Index(['series_id', 'time_idx', 'date', 'battery', 'barometric_pressure',
       'humidity', 'rain_meter', 'soil_moisture', 'temperature', 'wind_speed',
       'day_length', 'hour', 'dow', 'dom', 'hour_sin', 'hour_cos', 'dow_sin',
       'dow_cos'],
      dtype='object')

In [78]:
target = "battery"
observed_reals = ['barometric_pressure', 'humidity', 'rain_meter', 'soil_moisture', 'temperature', 'wind_speed', target]
known_reals = ['day_length', 'hour', 'dow', 'dom', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
training = TimeSeriesDataSet(
    # df[lambda x: x.time_idx < validation_start],
    df[df.time_idx <= validation_start],
    time_idx="time_idx",
    target=target,
    group_ids=["series_id"],
    min_encoder_length=max_encoder_length,  # Set equal to max_encoder_length
    max_encoder_length=max_encoder_length,
    min_prediction_length=max_prediction_length,  # Add this parameter
    max_prediction_length=max_prediction_length,
    static_categoricals=[],
    static_reals=[],
    time_varying_known_reals=known_reals,
    time_varying_unknown_reals=observed_reals,
    allow_missing_timesteps=True,
    target_normalizer=TorchNormalizer(method="standard"),
    add_relative_time_idx=True,  # Add this to help with temporal relationships
    add_target_scales=True       # Add this to help with scaling
)

# validation = TimeSeriesDataSet.from_dataset(training, df[df.time_idx > validation_start], predict=True, stop_randomization=True)

# Create validation dataset with predict=False first to check data
validation = TimeSeriesDataSet.from_dataset(
    training, 
    df[df.time_idx > validation_start - max_encoder_length],  # Include overlap for encoder
    predict=False,
    stop_randomization=True
)

# Check sizes before creating dataloaders
print(f"Training samples: {len(training)}")
print(f"Validation samples: {len(validation)}")

batch_size = 64  # set this between 32 to 128
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size * 2, num_workers=0)

Training samples: 17966
Validation samples: 18


## Configure & train TFT

In [79]:
early_stop_cb = EarlyStopping(monitor="val_loss", patience=5, mode="min")
ckpt_cb = ModelCheckpoint(monitor="val_loss", mode="min", save_top_k=1)
lr_monitor = LearningRateMonitor(logging_interval="epoch")  # log the learning rate

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=1e-3,
    loss=QuantileLoss(),              # probabilistic (P10/P50/P90). For point forecast, evaluate with MAE/RMSE below
    hidden_size=64,                   # model width
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=32,
    lstm_layers=1,
    output_size=7,                    # default for QuantileLoss (P10..P90). Will be inferred if not given.
    reduce_on_plateau_patience=3
)

trainer = Trainer(
    max_epochs=30,
    gradient_clip_val=0.1,
    callbacks=[early_stop_cb, ckpt_cb, lr_monitor],
    accelerator="auto",
    devices="auto",
    log_every_n_steps=50
)

trainer.fit(tft, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)
best_tft = tft.load_from_checkpoint(ckpt_cb.best_model_path)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Missing logger folder: /home/wqiaojoe/Development/ml-model-battery-levels-iot/notebooks/lightning_logs

   | Name                               | Type                            | Params | Mode 
------------------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0      | train
1  | logging_metrics                    | ModuleList                      | 0      | train
2  | input_embeddings                   | MultiEmbedding                  | 0      | train
3  | prescalers                         | ModuleDict                      | 1.2 K  | train
4  | static_variable_selection          | VariableSelectionNetwork        | 13.6 K | train
5  | encoder_variable_selection         | VariableSelectionNetwork        | 117 K  | train
6  | decoder_variable_selection         | VariableS

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

TypeError: The classmethod `TemporalFusionTransformer.load_from_checkpoint` cannot be called on an instance. Please call it on the class type and make sure the return value is used.